In [1]:
import cml.data_v1 as cmldata
 
# Sample in-code customization of spark configurations
#from pyspark import SparkContext
#SparkContext.setSystemProperty('spark.executor.cores', '1')
#SparkContext.setSystemProperty('spark.executor.memory', '2g')
 
CONNECTION_NAME = "pdnd-prod-dl-1"
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()
 
# Sample usage to run query through spark
EXAMPLE_SQL_QUERY = "show databases"
spark.sql(EXAMPLE_SQL_QUERY).show()

Setting spark.hadoop.yarn.resourcemanager.principal to maria.pantone


Spark Application Id:spark-6681e4a1e1144b5f845066675d35abcb


Hive Session ID = b5d540ce-a152-4ed9-b01b-2472ee3cab16


+--------------------+
|           namespace|
+--------------------+
|              app_io|
|  atlas_lineage_test|
|business_intellig...|
|            cashback|
|           checkiban|
|      data_engineers|
|       data_products|
|   data_products_dev|
|     data_scientists|
|       data_strategy|
|             default|
|      dl_anagrafiche|
|       dl_tassonomie|
|             dtd_bul|
|         dtd_bul_dev|
|             dtd_dev|
|             dtd_fse|
|         dtd_fse_dev|
|           dtd_istat|
|           dtd_pad26|
+--------------------+
only showing top 20 rows



In [2]:
import cml.data_v1 as cmldata
from pyspark.sql import functions as F

CONNECTION_NAME = "pdnd-prod-dl-1"
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()

csv_path = "file:///home/cdsw/20260223_total_analogico_to_check.csv"  # metti il tuo path

df_uuid_tmp = (
    spark.read
        .option("header", "true")    
        .option("inferSchema", "false")  
        .option("delimiter", ";")        
        .csv(csv_path)
)

df_uuid_tmp = (
    df_uuid_tmp
        .withColumn("uuid", F.trim(F.col("uuid")))
        .withColumn("birthDate", F.trim(F.col("birthDate")))
)

df_uuid_tmp.printSchema()
df_uuid_tmp.show(10, truncate=False)

Spark Application Id:spark-6681e4a1e1144b5f845066675d35abcb


root
 |-- uuid: string (nullable = true)
 |-- birthDate: string (nullable = true)

+---------------------------------------+----------+
|uuid                                   |birthDate |
+---------------------------------------+----------+
|PF-63556adc-ce70-4f2a-8d0b-7592116b8d2e|2026-12-30|
|PF-c6c91eb0-1973-463d-9177-b9b61ca9f61f|2026-12-30|
|PF-dc6de860-6296-4e05-8518-d9e249fdbbbc|2026-12-30|
|PF-00f2b6f9-741a-4038-aa24-4d3f5f372f32|2026-12-30|
|PF-16501b72-1249-489f-b6f7-5fc65e6cd94b|2026-12-30|
|PF-18cb256d-1253-4447-85fe-7d6e4420fc6a|2026-12-30|
|PF-2873bd2c-dc86-4317-b191-d4d73e102f2b|2026-12-30|
|PF-319d42ad-2a74-4126-958a-19db4d3aadcf|2026-12-30|
|PF-51c38cb1-f03d-4566-9da1-9d121dd2fc09|2026-12-30|
|PF-9a61c8b8-379b-4070-9afd-d83ec26663c1|2026-12-30|
+---------------------------------------+----------+
only showing top 10 rows



In [3]:
from IPython.display import display

display(df_uuid_tmp.limit(100).toPandas())

,uuid,birthDate
0,PF-63556adc-ce70-4f2a-8d0b-7592116b8d2e,2026-12-30
1,PF-c6c91eb0-1973-463d-9177-b9b61ca9f61f,2026-12-30
2,PF-dc6de860-6296-4e05-8518-d9e249fdbbbc,2026-12-30
3,PF-00f2b6f9-741a-4038-aa24-4d3f5f372f32,2026-12-30
4,PF-16501b72-1249-489f-b6f7-5fc65e6cd94b,2026-12-30
...,...,...
95,PF-928ddf3f-af58-48cf-b586-7060039ba9f6,2026-12-24
96,PF-b2ea5da3-5b31-4035-b3f3-cd972f3668cf,2026-12-24
97,PF-f88d1928-6e55-4641-8459-dfeb224e9fc5,2026-12-24
98,PF-062def23-013c-44e4-a7ff-064951c8c59d,2026-12-23


In [4]:
df_gold = spark.table("send.gold_notification_analytics")
df_gold.printSchema()

root
 |-- iun: string (nullable = true)
 |-- senderpaid: string (nullable = true)
 |-- sourcechannel: string (nullable = true)
 |-- sourcechanneldetails: string (nullable = true)
 |-- sourcechannel_visualization: string (nullable = true)
 |-- sourcechanneldetails_visualization: string (nullable = true)
 |-- taxonomycode: string (nullable = true)
 |-- pagopaintmode: string (nullable = true)
 |-- sentat: string (nullable = true)
 |-- type_notif: string (nullable = true)
 |-- recipients_size: integer (nullable = true)
 |-- actual_status: string (nullable = true)
 |-- status_tms_in_validation: string (nullable = true)
 |-- status_tms_accepted: string (nullable = true)
 |-- status_tms_refused: string (nullable = true)
 |-- status_tms_delivering: string (nullable = true)
 |-- status_tms_delivered: string (nullable = true)
 |-- status_tms_viewed: string (nullable = true)
 |-- status_tms_unreachable: string (nullable = true)
 |-- status_tms_effective_date: string (nullable = true)
 |-- status_

# match uuid-recipient e arricchimento: 
- iun
- senderpaid
- notificationsentat

In [5]:
df_uuid_tmp.createOrReplaceTempView("tmp_birthdates")
df_res = spark.sql("""
WITH exploded AS (
  SELECT
    g.iun,
    g.senderpaid,
    g.sentat AS notificationsentat,
    rec.recipientid
  FROM send.gold_notification_analytics g
  LATERAL VIEW explode(g.recipients) r AS rec
)
SELECT
  e.iun,
  e.senderpaid,
  e.notificationsentat
FROM exploded e
JOIN tmp_birthdates t
  ON trim(e.recipientid) = trim(t.uuid)
""")

df_res.show()

[Stage 4:>                                                          (0 + 1) / 1]

+--------------------+--------------------+--------------------+
|                 iun|          senderpaid|  notificationsentat|
+--------------------+--------------------+--------------------+
|AHLM-GHZJ-GVRZ-20...|1d8c397c-58ed-4bb...|2025-09-18T13:03:...|
|ALUW-PLWM-QRYA-20...|53b40136-65f2-424...|2025-12-15T16:27:...|
|AUYW-RDND-GJDQ-20...|76fd95f3-c1a1-410...|2025-04-11T08:16:...|
|AZKL-VQHL-MHUT-20...|8bdd616c-6130-4f8...|2024-06-27T14:49:...|
|DMVE-PXRP-UEXN-20...|8bdd616c-6130-4f8...|2024-06-27T14:10:...|
|DNQX-RYHK-LYAK-20...|d4062b08-f4c7-479...|2025-12-09T17:30:...|
|DUJV-QPAK-RLWU-20...|ea48f342-beea-48a...|2023-12-06T08:05:...|
|DYRD-KRHA-MRLG-20...|9dbec5d8-dcab-491...|2025-12-22T11:49:...|
|EAKA-ZELP-NYMT-20...|1d8c397c-58ed-4bb...|2025-11-26T11:52:...|
|EAPU-JAMG-NEXQ-20...|135100c9-d464-4ab...|2023-12-20T12:08:...|
|EMLZ-AEPQ-ZYKT-20...|508029b5-bcd0-40f...|2025-03-25T10:02:...|
|ENLN-LXRV-WXGW-20...|508029b5-bcd0-40f...|2025-03-25T08:58:...|
|EPNT-YQGX-UPVW-20...|1d8

# count distinct uuid-recipient e iun: 

In [6]:
df_count_recipients = spark.sql("""
WITH exploded AS (
  SELECT
    g.iun,
    g.senderpaid,
    g.sentat AS notificationsentat,
    rec.recipientid
  FROM send.gold_notification_analytics g
  LATERAL VIEW explode(g.recipients) r AS rec
)
SELECT
  COUNT(*)                      AS n_rows_join,
  COUNT(DISTINCT t.uuid)        AS n_distinct_uuid,
  COUNT(DISTINCT e.recipientid) AS n_distinct_recipientid,
  COUNT(DISTINCT e.iun) AS n_distinct_iun
FROM exploded e
JOIN tmp_birthdates t
  ON trim(e.recipientid) = trim(t.uuid)
""")

df_count_recipients.show(truncate=False)

[Stage 8:============================================>              (3 + 1) / 4]

+-----------+---------------+----------------------+--------------+
|n_rows_join|n_distinct_uuid|n_distinct_recipientid|n_distinct_iun|
+-----------+---------------+----------------------+--------------+
|19185      |10943          |10943                 |19185         |
+-----------+---------------+----------------------+--------------+



# Write count distinct uuid-recipient e iun: 

In [18]:
from datetime import datetime
import os

day = datetime.now().strftime("%Y%m%d")  # solo giorno

dir_path = f"/home/cdsw/exports/{day}"
os.makedirs(dir_path, exist_ok=True)

local_path = f"{dir_path}/count_cf_recipients_{day}.csv"


(df_count_recipients
  .toPandas()
  .to_csv(local_path, index=False, sep=";")
)

print("CSV locale scritto in:", local_path)

[Stage 103:==========================================>              (3 + 1) / 4]

CSV locale scritto in: /home/cdsw/exports/20260225/count_cf_recipients_20260225.csv


# selezione con campi ITA

In [9]:
from IPython.display import display

df_analisi_cf = spark.sql("""
WITH notifiche_gold AS (
  SELECT
    g.iun,
    g.senderpaid,
    g.sentat,
    g.actual_status,
    rec.recipientid
  FROM send.gold_notification_analytics g
  LATERAL VIEW explode(g.recipients) r AS rec
),
notifiche_silver AS (
  SELECT
    iun,
    sentAt             AS notificationSentAt,
    senderPaId,
    senderTaxId,
    senderDenomination
  FROM send.silver_notification   
)
SELECT
  e.iun                          AS iun,
  t.uuid                         AS cf_uuid,
  e.recipientid                  AS destinatario,
  e.senderpaid                   AS id_pa_mittente,
  n.senderPaId                   AS sender_pa_id,
  n.senderTaxId                  AS cf_pa_mittente,
  n.senderDenomination           AS pa_mittente,
  n.notificationSentAt           AS notificationSentAt,
  e.sentat                       AS data_invio,
  e.actual_status                AS status_actual,
  t.birthDate                    AS cf_data_nascita
FROM notifiche_gold e
JOIN tmp_birthdates t
  ON trim(e.recipientid) = trim(t.uuid)
LEFT JOIN notifiche_silver n
  ON e.iun = n.iun
ORDER BY iun
""")

df_analisi_cf.show(100, truncate=False)

[Stage 24:======================================================> (33 + 1) / 34]

+-------------------------+---------------------------------------+---------------------------------------+------------------------------------+------------------------------------+--------------+----------------------------------------+------------------------------+------------------------------+------------------+---------------+
|iun                      |cf_uuid                                |destinatario                           |id_pa_mittente                      |sender_pa_id                        |cf_pa_mittente|pa_mittente                             |notificationSentAt            |data_invio                    |status_actual     |cf_data_nascita|
+-------------------------+---------------------------------------+---------------------------------------+------------------------------------+------------------------------------+--------------+----------------------------------------+------------------------------+------------------------------+------------------+------------

# Selezione base richiesta JIRA analisi_cf_per recipient

In [ ]:
from IPython.display import display

df_per_recipient = spark.sql("""
WITH notifiche_gold AS (
  SELECT
    g.iun,
    g.actual_status,
    rec.recipientid
  FROM send.gold_notification_analytics g
  LATERAL VIEW explode(g.recipients) r AS rec
),
notifiche_silver AS (
  SELECT
    iun,
    sentAt             AS notificationSentAt,
    senderPaId,
    senderTaxId,
    senderDenomination
  FROM send.silver_notification
),
recipient_cf AS (
  SELECT
    trim(uuid) AS recipientId
  FROM tmp_birthdates
)
SELECT
  g.recipientid        AS recipientId,
  g.iun                AS iun,
  s.notificationSentAt AS notificationSentAt,
  s.senderPaId         AS senderPaId,
  s.senderTaxId        AS senderTaxId,
  s.senderDenomination AS senderDenomination,
  g.actual_status      AS actual_status
FROM notifiche_gold g
JOIN notifiche_silver s
  ON g.iun = s.iun
JOIN recipient_cf r
  ON trim(g.recipientid) = r.recipientId
ORDER BY recipientId, iun
""")

df_per_recipient.show(10, truncate=False)

# Anno e Mese 

In [10]:
from IPython.display import display

df_per_recipient = spark.sql("""
WITH notifiche_gold AS (
  SELECT
    g.iun,
    g.actual_status,
    rec.recipientid
  FROM send.gold_notification_analytics g
  LATERAL VIEW explode(g.recipients) r AS rec
),
notifiche_silver AS (
  SELECT
    iun,
    sentAt             AS notificationSentAt,
    senderPaId,
    senderTaxId,
    senderDenomination
  FROM send.silver_notification
),
recipient_cf AS (
  SELECT
    trim(uuid) AS recipientId
  FROM tmp_birthdates
)
SELECT
  g.recipientid        AS recipientId,
  g.iun                AS iun,
  s.notificationSentAt AS notificationSentAt,
  s.senderPaId         AS senderPaId,
  s.senderTaxId        AS senderTaxId,
  s.senderDenomination AS senderDenomination,
  g.actual_status      AS actual_status,
  SUBSTR(s.notificationSentAt, 1, 4) AS anno,
  SUBSTR(s.notificationSentAt, 6, 2) AS mese
FROM notifiche_gold g
JOIN notifiche_silver s
  ON g.iun = s.iun
JOIN recipient_cf r
  ON trim(g.recipientid) = r.recipientId
ORDER BY recipientId, iun
""")

df_per_recipient.show(10, truncate=False)
# display(df_per_recipient.limit(100).toPandas())

[Stage 30:======================================================> (65 + 2) / 67]

+---------------------------------------+-------------------------+------------------------------+------------------------------------+-----------+------------------+--------------+----+----+
|recipientId                            |iun                      |notificationSentAt            |senderPaId                          |senderTaxId|senderDenomination|actual_status |anno|mese|
+---------------------------------------+-------------------------+------------------------------+------------------------------------+-----------+------------------+--------------+----+----+
|PF-00094553-8549-47f4-b11e-ad5141283971|QGZM-RVHD-RPVA-202312-R-1|2023-12-29T21:17:49.561971319Z|9486a669-8afb-4f88-9417-f22f2239769b|00147990923|COMUNE DI CAGLIARI|EFFECTIVE_DATE|2023|12  |
|PF-00094553-8549-47f4-b11e-ad5141283971|ZADM-MPZL-GLUJ-202312-G-1|2023-12-29T21:15:54.969920923Z|9486a669-8afb-4f88-9417-f22f2239769b|00147990923|COMUNE DI CAGLIARI|EFFECTIVE_DATE|2023|12  |
|PF-000a7859-0018-4699-926b-ebb50fab0d16

# df filtered for: ('EFFECTIVE_DATE','DELIVERING','UNREACHABLE','CANCELLED','REFUSED')

In [17]:
from IPython.display import display

df_per_recipient.createOrReplaceTempView("per_recipient")

df_filtrata = spark.sql("""
WITH destinatari_ok AS (
  SELECT
    recipientId
  FROM per_recipient
  GROUP BY recipientId
  HAVING SUM(
    CASE
      WHEN actual_status NOT IN ('CANCELLED','DELIVERED','DELIVERING','EFFECTIVE_DATE','REFUSED')
        THEN 1
      ELSE 0
    END
  ) = 0
)
SELECT
  p.*
FROM per_recipient p
JOIN destinatari_ok d
  ON p.recipientId = d.recipientId
ORDER BY p.recipientId, p.iun
""")

df_filtrata.show(20, truncate=False)
# display(df_filtrata.limit(100).toPandas())

[Stage 101:>                                                        (0 + 1) / 1]

+---------------------------------------+-------------------------+------------------------------+------------------------------------+-----------+-------------------------+--------------+----+----+
|recipientId                            |iun                      |notificationSentAt            |senderPaId                          |senderTaxId|senderDenomination       |actual_status |anno|mese|
+---------------------------------------+-------------------------+------------------------------+------------------------------------+-----------+-------------------------+--------------+----+----+
|PF-00094553-8549-47f4-b11e-ad5141283971|QGZM-RVHD-RPVA-202312-R-1|2023-12-29T21:17:49.561971319Z|9486a669-8afb-4f88-9417-f22f2239769b|00147990923|COMUNE DI CAGLIARI       |EFFECTIVE_DATE|2023|12  |
|PF-00094553-8549-47f4-b11e-ad5141283971|ZADM-MPZL-GLUJ-202312-G-1|2023-12-29T21:15:54.969920923Z|9486a669-8afb-4f88-9417-f22f2239769b|00147990923|COMUNE DI CAGLIARI       |EFFECTIVE_DATE|2023|12  |
|PF-0

In [15]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Write to csv Results of Notification Enrichment

In [16]:
from datetime import datetime
import os

day = datetime.now().strftime("%Y%m%d")

dir_path = f"/home/cdsw/exports/{day}"
os.makedirs(dir_path, exist_ok=True)

filename = f"cf_notification_recipients_filtered_{day}.csv"
local_path = os.path.join(dir_path, filename)

if os.path.exists(local_path):
    os.remove(local_path)

(df_filtrata
  .toPandas()
  .to_csv(local_path, index=False, sep=";")
)

print("CSV locale scritto in:", local_path)

CSV locale scritto in: /home/cdsw/exports/20260225/cf_notification_recipients_filtered_20260225.csv


In [ ]:
df_gold = spark.table("send.silver_timeline")
df_gold.printSchema()